In [46]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import SABRModel
from scipy.optimize import minimize, LinearConstraint

# Configure pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:_.2f}')

In [47]:
def compute_option_surface(
    F: float, 
    K_list: list, 
    T_list: list, 
    alpha_list: list, 
    beta: float, 
    rho: float, 
    nu: float,
    r: float, 
    slide_scenario=None,
    slide_type: str = 'spot_only', 
    slide_compute: str = 'option_pnl',
    compute_bs_greeks: bool = True, 
    compute_model_greek: bool = False
) -> pd.DataFrame:
    """
    Computes SABR option prices and Greeks over a grid of strikes and maturities.
    
    Args:
        F: Forward price
        K_list: List of strike prices
        T_list: List of times to maturity (in years)
        alpha_list: List of alpha (volatility) parameters
        beta, rho, nu: SABR parameters
        r: Risk-free rate
        slide_scenario: List of spot bumps (optional)
        slide_type: 'spot_vol' or 'spot_only'
        slide_compute: PnL calculation type ('delta_hedged_pnl', 'option_pnl', 'delta_pnl')
        compute_bs_greeks: If True, returns Black-Scholes Greeks
        compute_model_greek: If True, returns SABR model Greeks
        
    Returns:
        DataFrame with rows for each (K, T, alpha) combination containing:
        - Input parameters: F, K, T, alpha, beta, rho, nu, r, option_type
        - IV: Implied volatility
        - price: Option price
        - Greeks: delta, gamma, vega, theta, vanna, volga
        - Model Greeks (if compute_model_greek=True): sabr_delta, sabr_gamma, sabr_vega, sabr_vanna, sabr_volga, sabr_theta
        - Slides: PnL or price differences for each slide scenario
        
    Note:
        Option type is determined automatically: call if K > F, put if K ≤ F
    """
    results = []
    
    for i in range(len(alpha_list)):
        alpha = alpha_list[i]
        T = T_list[i]
        for K in K_list:
            # Determine option type: call if K > F, put otherwise
            option_type = 'call' if K > F else 'put'
            
            result = SABRModel.compute_option(
                F=F, 
                K=K, 
                T=T, 
                alpha=alpha, 
                beta=beta, 
                rho=rho, 
                nu=nu,
                r=r, 
                option_type=option_type, 
                slide_scenario=slide_scenario,
                slide_type=slide_type, 
                slide_compute=slide_compute,
                compute_bs_greeks=compute_bs_greeks, 
                compute_model_greek=compute_model_greek)
                
            # Build row with inputs and results
            row = {
                'F': F,
                'K': K,
                'T': T,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'nu': nu,
                'r': r,
                'option_type': option_type,
            }
            
            # Add all result fields
            row.update(result)
            
            results.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns: inputs first, then IV and price, then greeks, then slides
    input_cols = ['F', 'K', 'T', 'alpha', 'beta', 'rho', 'nu', 'r', 'option_type']
    price_cols = ['IV', 'price']
    greek_cols = ['delta', 'gamma', 'vega', 'theta', 'vanna', 'volga']
    sabr_greek_cols = ['sabr_delta', 'sabr_gamma', 'sabr_vega', 'sabr_vanna', 'sabr_volga', 'sabr_theta']
    
    # Build column order
    col_order = input_cols + price_cols
    col_order += [c for c in greek_cols if c in df.columns]
    col_order += [c for c in sabr_greek_cols if c in df.columns]
    
    # Add slide columns (remaining columns)
    slide_cols = [c for c in df.columns if c not in col_order]
    col_order += slide_cols
    
    # Reorder dataframe
    df = df[col_order]
    
    return df

In [56]:
# Generate surface with slides
F = 100.0  # Forward price
K_max = 100
K_min = 50
K_step = 1
K_list = np.arange(K_min, K_max + K_step, K_step) # 9 strikes from 80 to 120 --- IGNORE ---
T_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]  # Times to maturity
alpha_list = [0.1, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2]  # Alpha (volatility) parameters

beta = 1
rho = -0.9
nu = 0.0001
r = 0.0

# Compute option surface with slides
slide_scenario = [-0.3, -0.1, -0.05, -0.04, -0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]
df_surface = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=slide_scenario,
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_surface['symbol'] = df_surface['K'].astype(str) + '_' + df_surface['T'].astype(str)
df_surface.set_index('symbol', inplace=True)
df =  df_surface.loc[:, slide_scenario]

In [60]:
def compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short):
    res_dict = dict()
    for long_leg_symbol in df.index:
        if long_leg_symbol == short_leg_symbol:
            continue
        selection_long = df.loc[long_leg_symbol]

        pos = selection_long > 0
        neg = selection_long < 0
        zero = selection_long == 0

        # Feasibility check on zero-slope indices
        if not (selection_short_scaled[zero] > 0).all():
            continue
            raise ValueError("Infeasible: selection <= 0 where selection_long == 0")

        L = (-selection_short_scaled[pos] / selection_long[pos]).max() if pos.any() else -np.inf
        U = (-selection_short_scaled[neg] / selection_long[neg]).min() if neg.any() else  np.inf

        if L >= U:
            continue
            raise ValueError("Infeasible: lower bound exceeds upper bound")

        q_long = L + 1e-9 # or any small epsilon
        selection_long_scaled = selection_long * q_long
        selection_combined_scaled = selection_short_scaled + selection_long_scaled 
        res_dict[(short_leg_symbol, long_leg_symbol)] = {'q_short': q_short, 'q_long': q_long, **selection_combined_scaled.to_dict()}
    return res_dict

In [ ]:
# compute combinaison
res = dict()
for short_leg_symbol in df.index:

    selection_short = df.loc[short_leg_symbol]
    q_short = -30_000_000 / selection_short[-0.3]
    selection_short_scaled = selection_short * q_short

    res_temp = compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short)
    res.update(res_temp)

In [ ]:
# display and filter results
res_df = pd.DataFrame(res).T
res_df = res_df.reset_index(names=['short_symbol', 'long_symbol'])
df_surface['ask_price'] = df_surface['vega'] + df_surface['price']
df_surface['bid_price'] = -df_surface['vega'] + df_surface['price']
res_df['bid_price'] = res_df['short_symbol'].map(df_surface['bid_price'])
res_df['ask_price'] = res_df['long_symbol'].map(df_surface['ask_price'])
res_df['e_pnl'] = (res_df['q_short'] * res_df['bid_price'] * -1) - (res_df['q_long'] * res_df['ask_price'])
res_df = res_df[res_df['e_pnl']>0].sort_values('e_pnl', ascending=False).reset_index(drop=True)
res_df['ratio'] = -res_df['q_short'] / res_df['q_long']
res_df = res_df.loc[res_df['ratio']<2]

In [88]:
# optimized function
def compute_hedges_fast(selection_short_scaled_arr, values, index_arr, short_idx,
                        short_leg_symbol, q_short, columns):
    """
    Vectorized computation of hedges for a single short leg against all long legs.
    
    Parameters
    ----------
    selection_short_scaled_arr : np.ndarray, shape (n_cols,)
    values : np.ndarray, shape (n_rows, n_cols)  -- df.values
    index_arr : np.ndarray of object, shape (n_rows,) -- df.index.values
    short_idx : int -- row index of short leg
    """
    n_rows, n_cols = values.shape

    # Mask out the short leg row itself
    row_mask = np.ones(n_rows, dtype=bool)
    row_mask[short_idx] = False

    long_vals = values[row_mask]                    # (n_rows-1, n_cols)
    long_symbols = index_arr[row_mask]              # (n_rows-1,)

    pos_mask = long_vals > 0                        # (n_rows-1, n_cols)
    neg_mask = long_vals < 0
    zero_mask = ~(pos_mask | neg_mask)              # equivalent to long_vals == 0

    # Feasibility: where long == 0, short_scaled must be > 0
    short_pos = selection_short_scaled_arr > 0      # (n_cols,)
    # For each row, every zero-slope column must have short_pos True
    infeasible_zero = (zero_mask & ~short_pos).any(axis=1)
    if infeasible_zero.any():
        bad = long_symbols[infeasible_zero][0]
        raise ValueError(
            f"Infeasible: selection <= 0 where selection_long == 0 "
            f"(short={short_leg_symbol}, long={bad})"
        )

    # Compute -short_scaled / long_vals safely; we'll mask out invalid entries
    with np.errstate(divide='ignore', invalid='ignore'):
        ratios = -selection_short_scaled_arr[None, :] / long_vals  # (n_rows-1, n_cols)

    # L = max over pos entries; if no pos, -inf
    pos_ratios = np.where(pos_mask, ratios, -np.inf)
    L = pos_ratios.max(axis=1)                      # (n_rows-1,)

    # U = min over neg entries; if no neg, +inf
    neg_ratios = np.where(neg_mask, ratios, np.inf)
    U = neg_ratios.min(axis=1)                      # (n_rows-1,)

    feasible = L < U                                # (n_rows-1,)

    if not feasible.any():
        return {}

    # Filter to feasible rows only
    L_f = L[feasible]
    long_vals_f = long_vals[feasible]
    long_symbols_f = long_symbols[feasible]

    q_long = L_f + 1e-9                             # (k,)
    # selection_long_scaled = long_vals * q_long  (broadcast over columns)
    long_scaled = long_vals_f * q_long[:, None]     # (k, n_cols)
    combined = selection_short_scaled_arr[None, :] + long_scaled  # (k, n_cols)

    # Build result dict
    res = {}
    cols_list = list(columns)
    for i, long_sym in enumerate(long_symbols_f):
        row = combined[i]
        d = {'q_short': q_short, 'q_long': q_long[i]}
        d.update(zip(cols_list, row))
        res[(short_leg_symbol, long_sym)] = d
    return res


def compute_all_hedges_fast(df):
    """
    Drop-in replacement for the outer loop.
    Returns the same `res` dict the original code builds.
    """
    values = np.ascontiguousarray(df.values, dtype=np.float64)
    index_arr = df.index.to_numpy()
    columns = df.columns

    # Locate the column corresponding to -0.3 once
    # (matches df.loc[symbol][-0.3] semantics, i.e. label-based lookup)
    col_neg03_idx = df.columns.get_loc(-0.3)

    res = {}
    for short_idx in range(values.shape[0]):
        short_leg_symbol = index_arr[short_idx]
        selection_short = values[short_idx]

        q_short = -60_000_000 / selection_short[col_neg03_idx]
        selection_short_scaled = selection_short * q_short

        res_temp = compute_hedges_fast(
            selection_short_scaled, values, index_arr,
            short_idx, short_leg_symbol, q_short, columns,
        )
        res.update(res_temp)
    return res


def compute_hedges_for_product(df, target_symbol):
    """
    Compute the best hedges for a single target product (used as the short leg)
    against all other products as long legs.
    """
    values = np.ascontiguousarray(df.values, dtype=np.float64)
    index_arr = df.index.to_numpy()
    columns = df.columns

    col_neg03_idx = df.columns.get_loc(-0.3)

    # Locate the row for the target product
    # Using np.where avoids issues if the index isn't a pandas Index lookup
    matches = np.where(index_arr == target_symbol)[0]
    if len(matches) == 0:
        raise KeyError(f"Symbol {target_symbol!r} not found in df.index")
    short_idx = int(matches[0])

    selection_short = values[short_idx]
    q_short = -60_000_000 / selection_short[col_neg03_idx]
    selection_short_scaled = selection_short * q_short

    return compute_hedges_fast(
        selection_short_scaled, values, index_arr,
        short_idx, target_symbol, q_short, columns,
    )

# Usage:
# res = compute_hedges_for_product(df, "MY_SYMBOL")

# Usage:
#res = compute_all_hedges_fast(df)
#pd.DataFrame(res).T